
## 6.1 🧠 Conceptual Questions
### 1. **The No-Communication Theorem:** Bob cannot tell that teleportation happened by just looking at his qubit before receiving Alice's classical bits. Why?

**Ans** When Alice performs measurement, two qubits are collasped into one of four possible outcomes (00, 01, 10, or 11), each occurring with equal probability. Depending on which outcome Alice gets, Bob’s qubit shifts into a specific corresponding state.
#### Measurement Outcomes and Corresponding States

* **If Alice measures `00`** $\rightarrow$ Bob's qubit is exactly in the target state:   
**Bob applies:** Nothing (Identity gate $I$).

* **If Alice measures `01`** $\rightarrow$ Bob's qubit is bit-flipped:   
**Bob applies:** $X$ gate to fix the bit-flip.

* **If Alice measures `10`** $\rightarrow$ Bob's qubit is phase-flipped:   
**Bob applies:** $Z$ gate to fix the phase-flip.

* **If Alice measures `11`** $\rightarrow$ Bob's qubit is both bit- and phase-flipped:   
**Bob applies:** $X$ then $Z$ gate  to fix both flips.  

### 2. **No-Cloning:** After teleportation, does Alice still have the original state $|\psi\rangle$? Why or why not?  
**Ans** No, Alice does not keep the original state. Quantum teleportation is a state transfer, it is not a copy operation, it is a move operation.

* **Before the test:** The quantum state lives on Alice's first qubit.
* **The test:** Alice measures both of her qubits together. This instantly forces them to collapse into definite, classical states (`0` or `1`).
* **The result:** This measurement completely destroys the quantum superposition on Alice's side. Her qubits are left as basic classical bits, and the original information is wiped entirely from her system.

### 3. **Classical bits required:** The protocol uses 2 classical bits. Is it possible to teleport using just 1 classical bit? Why or why not?
**Ans** No, it is not possible to teleport using only 1 classical bit. A minimum of 2 classical bits is needed.  
When Alice measures her two qubits, they collapse into one of **4 possible combinations** (`00`, `01`, `10`, or `11`). To tell Bob exactly which correction gate to use, she must send enough data to cover all 4 options.

* **1 classical bit** can only carry **2 messages** (`0` or `1`).
* **2 classical bits** can carry **4 messages** (`00`, `01`, `10`, or `11`).  

### 4. **The role of entanglement:** Could the protocol work if Alice and Bob shared a **separable** (non-entangled) pair instead? What would happen?
**Ans** No, the it will completely fail. Without quantum entanglement, quantum state information cannot be transferred from Alice to Bob.

If they use a separable (unentangled) pair Alice's qubit and Bob's qubit are completely independent systems. There is no mathematical or physical correlation connecting them.

### 5. **Fidelity:** If the entangled pair were not a perfect Bell state (e.g., slightly noisy), what would happen to the teleported state?
**Ans** Teleported state received by Bob will be distorted. The quantum fidelity will drop.

## 6.2 💻 Coding Exercises

**Exercise 1 — Teleport a different state:**  
Modify Step 1 to teleport the $|+\rangle$ state (just a Hadamard applied to $|0\rangle$). Change the verification in Step 5 accordingly. Does the success rate remain high?

In [2]:
# ── Exercise scaffold: Teleport |+⟩ state ─────────────────────────────────
from qiskit import QuantumCircuit, QuantumRegister, ClassicalRegister
from qiskit_aer import AerSimulator
from collections import Counter

alice2  = QuantumRegister(1, name='alice')
ep2     = QuantumRegister(1, name='ep')
bob2    = QuantumRegister(1, name='bob')
alice_c2 = ClassicalRegister(1, name='alicec')
ep_c2   = ClassicalRegister(1, name='epc')
bob_c2  = ClassicalRegister(1, name='bobc')

qc2 = QuantumCircuit(alice2, ep2, bob2, alice_c2, ep_c2, bob_c2)

# Step 0: Create the shared Bell pair
qc2.h(ep2)
qc2.cx(ep2, bob2)
qc2.barrier()

# Step 1: Prepare the |+⟩ state on Alice's qubit
qc2.h(alice2)  #  Hadamard to put Alice's qubit into |+⟩
qc2.barrier()

# Steps 2–4: Standard Bell State Measurement & Corrections
qc2.cx(alice2, ep2)
qc2.h(alice2)
qc2.measure(alice2, alice_c2)
qc2.measure(ep2, ep_c2)
qc2.barrier()

# Apply Bob's corrections based on classical registers
with qc2.if_test((alice_c2, 1)):
    qc2.z(bob2)
with qc2.if_test((ep_c2, 1)):
    qc2.x(bob2)
qc2.barrier()

# Step 5: Verify by applying the inverse preparation and measuring
qc2.h(bob2)    # The inverse of H is H itself! Brings |+⟩ back to |0⟩
qc2.measure(bob2, bob_c2)

# ── Simulation & Verification Execution Block ─────────────────────────────
simulator2 = AerSimulator()
job2 = simulator2.run(qc2, shots=1000)
counts2 = job2.result().get_counts(qc2)

# Analyze Bob's success rate
bob_0_count = 0
bob_1_count = 0

for outcome, count in counts2.items():
    # Outcome keys split by space represent registers: [bobc epc alicec]
    bob_bit = outcome.split()[0]
    if bob_bit == '0':
        bob_0_count += count
    else:
        bob_1_count += count

print("\n" + "="*45)
print("          EXERCISE 1 RESULTS")
print("="*45)
print(f"Bob measured |0⟩ (Success): {bob_0_count} times")
print(f"Bob measured |1⟩ (Failure): {bob_1_count} times")
print(f"👉 Success Rate: {(bob_0_count / 1000) * 100:.1f}%")
print("="*45)


          EXERCISE 1 RESULTS
Bob measured |0⟩ (Success): 1000 times
Bob measured |1⟩ (Failure): 0 times
👉 Success Rate: 100.0%


**Exercise 2 — Teleport $|0\rangle$:**  
What if Alice's qubit starts as $|0\rangle$ with no extra gates applied? What is the expected outcome for Bob? (You should be able to work this out mathematically *and* verify it with the circuit.)

**Ans** Bob's qubit will inherit the state $|0\rangle$ perfectly after he applies his corrections.  
Since Alice applied zero gates to prepare $|0\rangle$, the inverse is also zero gates (the Identity operation). Therefore, when Bob directly measures his qubit, it should collapse into the $|0\rangle$ state with a 100% probability (returning a classical bit value of 0).

In [3]:
# ── Exercise 2: Teleporting the |0⟩ state ─────────────────────────────────
from qiskit import QuantumCircuit, QuantumRegister, ClassicalRegister
from qiskit_aer import AerSimulator
from collections import Counter

alice3   = QuantumRegister(1, name='alice')
ep3      = QuantumRegister(1, name='ep')
bob3     = QuantumRegister(1, name='bob')
alice_c3 = ClassicalRegister(1, name='alicec')
ep_c3    = ClassicalRegister(1, name='epc')
bob_c3   = ClassicalRegister(1, name='bobc')

qc3 = QuantumCircuit(alice3, ep3, bob3, alice_c3, ep_c3, bob_c3)

# Step 0: Create the shared Bell pair
qc3.h(ep3)
qc3.cx(ep3, bob3)
qc3.barrier()

# Step 1: Prepare |0⟩ state on Alice's qubit 
# We intentionally leave this empty because a fresh qubit automatically starts in |0⟩
qc3.barrier()

# Steps 2–4: Standard Bell State Measurement & Corrections
qc3.cx(alice3, ep3)
qc3.h(alice3)
qc3.measure(alice3, alice_c3)
qc3.measure(ep3, ep_c3)
qc3.barrier()

# Bob's conditional corrections
with qc3.if_test((alice_c3, 1)):
    qc3.z(bob3)
with qc3.if_test((ep_c3, 1)):
    qc3.x(bob3)
qc3.barrier()

# Step 5: Verify by measuring Bob's qubit directly 
# No inverse gate is needed because the inverse of doing nothing is doing nothing!
qc3.measure(bob3, bob_c3)

# ── Simulation Run ────────────────────────────────────────────────────────
simulator3 = AerSimulator()
job3 = simulator3.run(qc3, shots=1000)
counts3 = job3.result().get_counts(qc3)

# Count outcomes for Bob's verification bit
bob_0_count = 0
bob_1_count = 0

for outcome, count in counts3.items():
    # Outcome keys split by space represent registers: [bobc epc alicec]
    bob_bit = outcome.split()[0]
    if bob_bit == '0':
        bob_0_count += count
    else:
        bob_1_count += count

print("\n" + "="*45)
print("          EXERCISE 2 RESULTS")
print("="*45)
print(f"Bob measured |0⟩ (Success): {bob_0_count} times")
print(f"Bob measured |1⟩ (Failure): {bob_1_count} times")
print(f"👉 Teleportation Fidelity: {(bob_0_count / 1000) * 100:.1f}%")
print("="*45)


          EXERCISE 2 RESULTS
Bob measured |0⟩ (Success): 1000 times
Bob measured |1⟩ (Failure): 0 times
👉 Teleportation Fidelity: 100.0%


**Exercise 3 — What happens without corrections?:**  
Remove Bob's conditional gates (the `.c_if` gates) and re-run. What does the histogram look like? What does this tell you about why the corrections are necessary?
**Ans** Bob will bypass the restoration phase. Instead of bringing his qubit back into alignment with Alice's original state, he will simply measure it exactly as it was left immediately after Alice’s measurement.


In [4]:
# ── Exercise 3: Teleportation WITHOUT Corrections ─────────────────────────
from qiskit import QuantumCircuit, QuantumRegister, ClassicalRegister
from qiskit_aer import AerSimulator
import math

alice4   = QuantumRegister(1, name='alice')
ep4      = QuantumRegister(1, name='ep')
bob4     = QuantumRegister(1, name='bob')
alice_c4 = ClassicalRegister(1, name='alicec')
ep_c4    = ClassicalRegister(1, name='epc')
bob_c4   = ClassicalRegister(1, name='bobc')

qc4 = QuantumCircuit(alice4, ep4, bob4, alice_c4, ep_c4, bob_c4)

# Step 0: Create the shared Bell pair
qc4.h(ep4)
qc4.cx(ep4, bob4)
qc4.barrier()

# Step 1: Prepare the original state to teleport (same as your notebook example)
qc4.reset(alice4)
qc4.ry(math.radians(45), alice4)
qc4.h(alice4)
qc4.barrier()

# Steps 2-3: Bell State Measurement on Alice's side
qc4.cx(alice4, ep4)
qc4.h(alice4)
qc4.measure(alice4, alice_c4)
qc4.measure(ep4, ep_c4)
qc4.barrier()

# ❌ STEP 4: REMOVED BOB'S CONDITIONAL CORRECTIONS
# We skip the conditional X and Z gates entirely to see what happens.
qc4.barrier()

# Step 5: Partial inverse to verify teleportation
qc4.ry(math.radians(-45), bob4) 
qc4.measure(bob4, bob_c4)

# ── Simulation Run ────────────────────────────────────────────────────────
simulator4 = AerSimulator()
job4 = simulator4.run(qc4, shots=1000)
counts4 = job4.result().get_counts(qc4)

# Count outcomes for Bob's verification bit
bob_0_count = 0
bob_1_count = 0

for outcome, count in counts4.items():
    # Outcome keys split by space represent registers: [bobc epc alicec]
    bob_bit = outcome.split()[0]
    if bob_bit == '0':
        bob_0_count += count
    else:
        bob_1_count += count

print("\n" + "="*45)
print("          EXERCISE 3 RESULTS (NO CORRECTIONS)")
print("="*45)
print(f"Bob measured |0⟩ (Success): {bob_0_count} times")
print(f"Bob measured |1⟩ (Failure): {bob_1_count} times")
print(f"👉 Success Rate: {(bob_0_count / 1000) * 100:.1f}%")
print("="*45)


          EXERCISE 3 RESULTS (NO CORRECTIONS)
Bob measured |0⟩ (Success): 495 times
Bob measured |1⟩ (Failure): 505 times
👉 Success Rate: 49.5%
